In [ ]:
# Load convolve_im_freq from freq_1.ipynb 
from IPython.display import clear_output
%run -n freq_1.ipynb 
clear_output(wait=False) 

import os
import numpy as np
import utils
import matplotlib.pyplot as plt 

In [ ]:
def create_kernel(im):
    """
    Creates a filter kernel in the frequency domain.

    Args:
        im: np.array of shape [H, W]
    Returns:
        kernel: np.array of shape [H, W]
    """
    spectrum = np.abs(np.fft.fftshift(np.fft.fft2(im)))
    height, width = im.shape
    center_row, center_col = height // 2, width // 2

    yy, xx = np.ogrid[:height, :width]
    search_radius = max(8, min(height, width) // 20)
    spectrum[center_row - search_radius:center_row + search_radius + 1,
             center_col - search_radius:center_col + search_radius + 1] = 0
    horizontal_spectrum = spectrum[center_row].copy()
    search_start = center_col + search_radius
    peak_col = search_start + np.argmax(horizontal_spectrum[search_start:])

    kernel = np.ones_like(im, dtype=float)
    notch_radius = max(3, min(height, width) // 100)
    frequency_step = peak_col - center_col
    max_harmonic = width // (2 * frequency_step)
    for harmonic in range(1, max_harmonic + 1):
        for notch_col in (center_col - harmonic * frequency_step,
                          center_col + harmonic * frequency_step):
            if 0 <= notch_col < width:
                kernel[(yy - center_row) ** 2 +
                       (xx - notch_col) ** 2 <= notch_radius ** 2] = 0.0
    kernel = np.fft.ifftshift(kernel)
    return kernel

In [ ]:
if __name__ == "__main__":
    # Changing this code should not be needed!
    
    impath = os.path.join("images", "noisy_moon.png")
    im = utils.read_im(impath)

    kernel = create_kernel(im)        
    # Note that this function is from your own implementation of convolve_im_freq from freq_1.ipynb
    im_filtered = convolve_im_freq(im, kernel, True) 
    
    utils.save_im("moon_filtered.png", utils.normalize(im_filtered))